# LexRAG: serwer modelu w Google Colab (Gemma 4 12B na karcie T4)

Ten notatnik robi jedno: uruchamia Ollamę z modelem **Gemma 4 12B** na karcie Colaba i wystawia ją przez tunel,
żeby LexRAG na NAS-ie (karta 6 GB, na której 12B się nie mieści) mógł wysyłać do niej wywołania modelu.
Wyszukiwanie, indeks i dokumenty zostają na NAS-ie; do Colaba lecą tylko prompty (pytanie + wycięte przepisy).

**Zasady:**
- karta **T4 wystarcza** (model ~10 GB w pamięci karty; tu nie działa proces pomiaru, więc 12,7 GB RAM maszyny nie przeszkadza);
- używaj tylko z **aktami publicznymi**; przy tym ustawieniu nie wgrywaj na NAS dokumentów wewnętrznych, bo ich fragmenty poszłyby do chmury;
- adres tunelu jest losowy i zmienia się przy każdym uruchomieniu; nie ma hasła, więc po pracy zatrzymaj komórkę 3;
- sesja Colab Pro żyje do kilkunastu godzin; to narzędzie do testów UI, nie do produkcji (regulamin Colaba zabrania stałego hostowania usług).

Po uruchomieniu komórki 2 skopiuj wypisany adres do `.env` LexRAG-a na NAS-ie:

```
LEXRAG_LLM_BASE_URL=https://<adres>.trycloudflare.com/v1
LEXRAG_LLM_API=ollama
LEXRAG_LLM_MODEL=gemma4:12b
```

i zrestartuj kontener `lexrag-app` (`docker compose up -d lexrag`). Kontener `ollama` na NAS-ie może zostać wyłączony.


In [ ]:
# 1. Ollama + Gemma 4 12B (około 5 minut: instalator + pobranie 7,6 GB)
!apt-get -qq install -y zstd > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh
import os, subprocess, time, urllib.request
env = dict(os.environ, OLLAMA_HOST="0.0.0.0:11434", OLLAMA_ORIGINS="*", OLLAMA_KEEP_ALIVE="-1", OLLAMA_NUM_PARALLEL="2")
ollama_proc = subprocess.Popen(["ollama", "serve"], env=env, stdout=open("/content/ollama.log", "w"), stderr=subprocess.STDOUT)
for _ in range(90):
    try:
        urllib.request.urlopen("http://127.0.0.1:11434/"); break
    except Exception:
        time.sleep(1)
!ollama pull gemma4:12b
!ollama run gemma4:12b "Odpowiedz jednym słowem: gotowe?" --nowordwrap
!nvidia-smi --query-gpu=name,memory.used,memory.total --format=csv


In [ ]:
# 2. Tunel do Ollamy (cloudflared, bez konta) – wypisuje adres do wpisania na NAS-ie
!wget -q -O /content/cloudflared.deb https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb && dpkg -i /content/cloudflared.deb > /dev/null
import re, subprocess, time, urllib.request, json
tunel = subprocess.Popen(["cloudflared", "tunnel", "--url", "http://127.0.0.1:11434", "--no-autoupdate"],
                         stdout=open("/content/tunel.log", "w"), stderr=subprocess.STDOUT)
adres = ""
for _ in range(60):
    time.sleep(2)
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", open("/content/tunel.log").read())
    if m:
        adres = m.group(0); break
assert adres, "tunel nie wystartował – zobacz /content/tunel.log"
modele = json.load(urllib.request.urlopen(adres + "/api/tags"))
print("Modele widoczne przez tunel:", [m["name"] for m in modele.get("models", [])])
print("\nWpisz na NAS-ie do .env:")
print(f"LEXRAG_LLM_BASE_URL={adres}/v1")
print("LEXRAG_LLM_API=ollama")
print("LEXRAG_LLM_MODEL=gemma4:12b")
print("\nTest z laptopa:  python -m lexrag llm-test   (po ustawieniu tych samych zmiennych)")


In [ ]:
# 3. Podtrzymanie sesji – komórka działa, dopóki jej nie zatrzymasz (maks. 11 h); zatrzymanie zamyka tunel
import time, urllib.request, datetime
for i in range(11 * 12):
    try:
        urllib.request.urlopen("http://127.0.0.1:11434/api/tags", timeout=10)
        print(datetime.datetime.now().strftime("%H:%M"), "serwer modelu działa, adres:", adres, flush=True)
    except Exception as e:
        print("Ollama nie odpowiada:", e, flush=True)
    time.sleep(300)
tunel.terminate()
